In [17]:
import os
import sys
from pydantic import BaseModel, Field
from typing import Optional

# Define the base directory for the mock 'app' package
# Using /content for more reliable module resolution in Colab
app_base_dir = "/content"
app_dir = os.path.join(app_base_dir, 'app')

# Add the base directory to sys.path so 'app' can be imported
if app_base_dir not in sys.path:
    sys.path.insert(0, app_base_dir)

# Create 'app' directory if it doesn't exist
if not os.path.isdir(app_dir):
    os.makedirs(app_dir)

# Ensure __init__.py exists to make 'app' a package
init_py_path = os.path.join(app_dir, '__init__.py')
if not os.path.exists(init_py_path):
    with open(init_py_path, 'w') as f:
        f.write("# This makes 'app' a Python package\n")

# Ensure a mock main.py exists with necessary variables for testing
main_py_path = os.path.join(app_dir, 'main.py')

# Overwrite main.py to ensure the latest version with Pydantic models is always used
with open(main_py_path, 'w') as f:
    f.write("""
from fastapi import FastAPI
from pydantic import BaseModel, Field
import os

app = FastAPI()
MODEL_PATH = os.path.join(os.path.dirname(__file__), "mock_model.pkl") # Define a mock model path for testing

# Define Pydantic model for prediction request payload
class PredictionRequest(BaseModel):
    customer_id: str
    total_spend: float = Field(..., gt=0) # Must be greater than 0
    order_count: int = Field(..., gt=0)   # Must be greater than 0
    ticket_count: int = Field(..., ge=0)  # Must be greater than or equal to 0
    web_sessions: int = Field(..., ge=0) # Must be greater than or equal to 0

@app.get("/health")
def health_check():
    return {"status": "healthy"}

@app.post("/predict")
def predict(request: PredictionRequest):
    # This is a mock implementation for testing purposes
    # It now uses the validated request data
    return {
        "customer_id": request.customer_id,
        "churn_probability": 0.5,
        "predicted_class": 0
    }
""")


In [19]:
%%writefile /content/test_app.py

import os
import sys
import pytest
from fastapi.testclient import TestClient
import pickle
import numpy as np
import importlib # Import importlib for module reloading

# Add the base directory to sys.path so 'app' can be imported
# Ensure /content is in sys.path for module discovery if not already added
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# If app.main module was previously imported, reload it to pick up changes from disk
if 'app.main' in sys.modules:
    importlib.reload(sys.modules['app.main'])

from app.main import app, MODEL_PATH

# Define MockModel at module level for pickling
class MockModel:
    def predict_proba(self, X):
        # Return standard static binary classification distribution shape mapping array
        return np.array([[0.80, 0.20]] * len(X))

# Generate mock model file weights specifically for test isolation execution stability
@pytest.fixture(scope="session", autouse=True)
def setup_mock_model_weights():
    if not os.path.exists(MODEL_PATH):
        mock_payload = {'model': MockModel(), 'scaler': None}
        with open(MODEL_PATH, "wb") as f:
            pickle.dump(mock_payload, f)
    yield

client = TestClient(app)

def test_health_endpoint_return_status():
    """Test 1: Verify operational system application state properties."""
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["status"] == "healthy"

def test_predict_single_valid_payload():
    """Test 2: Verify scoring pipeline handles healthy validation features correctly."""
    valid_payload = {
        "customer_id": "CUST-9999",
        "total_spend": 150.00,
        "order_count": 3,
        "ticket_count": 0,
        "web_sessions": 25
    }
    response = client.post("/predict", json=valid_payload)
    assert response.status_code == 200
    data = response.json()
    assert data["customer_id"] == "CUST-9999"
    assert "churn_probability" in data
    assert "predicted_class" in data

def test_predict_single_invalid_validation_types():
    """Test 3: Enforce input type constraints via Pydantic error handlers."""
    invalid_payload = {
        "customer_id": "CUST-FAULT",
        "total_spend": -50.00,  # Invalid negative spend validation exception check
        "order_count": "three", # Invalid datatype string instead of integer
        "ticket_count": 1,
        "web_sessions": 12
    }
    response = client.post("/predict", json=invalid_payload)
    assert response.status_code == 422  # Request parsing schema validation error status


Overwriting /content/test_app.py


In [12]:
import pytest

# Run pytest to discover and execute tests in /content/test_app.py
pytest.main([])

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: anyio-4.13.0, langsmith-0.8.12, typeguard-4.5.2
collected 3 items

test_app.py EEE                                                          [100%]

==================================== ERRORS ====================================
_____________ ERROR at setup of test_health_endpoint_return_status _____________

    @pytest.fixture(scope="session", autouse=True)
    def setup_mock_model_weights():
        if not os.path.exists(MODEL_PATH):
            class MockModel:
                def predict_proba(self, X):
                    # Return standard static binary classification distribution shape mapping array
                    return np.array([[0.80, 0.20]] * len(X))
    
            mock_payload = {'model': MockModel(), 'scaler': None}
            with open(MODEL_PATH, "wb") as f:
>               pickle.dump(mock_payloa

<ExitCode.TESTS_FAILED: 1>

In [14]:
import pytest

# Run pytest again to discover and execute tests in /content/test_app.py
pytest.main([])

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: anyio-4.13.0, langsmith-0.8.12, typeguard-4.5.2
collected 3 items

test_app.py .FF                                                          [100%]

=================================== FAILURES ===================================
______________________ test_predict_single_valid_payload _______________________

    
    def test_predict_single_valid_payload():
        """Test 2: Verify scoring pipeline handles healthy validation features correctly."""
        valid_payload = {
            "customer_id": "CUST-9999",
            "total_spend": 150.00,
            "order_count": 3,
            "ticket_count": 0,
            "web_sessions": 25
        }
        response = client.post("/predict", json=valid_payload)
        assert response.status_code == 200
>       data = response.json()
E       AssertionError: assert 'test_

<ExitCode.TESTS_FAILED: 1>

In [16]:
import pytest

# Run pytest again after updating app/main.py
pytest.main([])

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: anyio-4.13.0, langsmith-0.8.12, typeguard-4.5.2
collected 3 items

test_app.py .FF                                                          [100%]

=================================== FAILURES ===================================
______________________ test_predict_single_valid_payload _______________________

    
    def test_predict_single_valid_payload():
        """Test 2: Verify scoring pipeline handles healthy validation features correctly."""
        valid_payload = {
            "customer_id": "CUST-9999",
            "total_spend": 150.00,
            "order_count": 3,
            "ticket_count": 0,
            "web_sessions": 25
        }
        response = client.post("/predict", json=valid_payload)
        assert response.status_code == 200
>       data = response.json()
E       AssertionError: assert 'test_

<ExitCode.TESTS_FAILED: 1>

In [18]:
import pytest

# Re-run pytest after ensuring app/main.py is updated with Pydantic validation
pytest.main([])

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: anyio-4.13.0, langsmith-0.8.12, typeguard-4.5.2
collected 3 items

test_app.py .FF                                                          [100%]

=================================== FAILURES ===================================
______________________ test_predict_single_valid_payload _______________________

    
    def test_predict_single_valid_payload():
        """Test 2: Verify scoring pipeline handles healthy validation features correctly."""
        valid_payload = {
            "customer_id": "CUST-9999",
            "total_spend": 150.00,
            "order_count": 3,
            "ticket_count": 0,
            "web_sessions": 25
        }
        response = client.post("/predict", json=valid_payload)
        assert response.status_code == 200
>       data = response.json()
E       AssertionError: assert 'test_

<ExitCode.TESTS_FAILED: 1>